# gen.py
Generates all segmentations for the models in the base models directory

## Change directory to base of src, fusionLearning/fusionLearning

In [1]:
import os
from pathlib import Path

root_is_cwd = os.getcwd().endswith("fusionLearning")

if not root_is_cwd:
    os.chdir(Path().resolve().parent)
    print("Changed to root directory")
    root_is_cwd = True
else:
    print("Already in root directory")

print(os.getcwd())

Changed to root directory
C:\Users\GAMER01\codeproj\fusionLearning\fusionLearning


In [3]:
from config import BASE_MODELS_SEGMENTATIONS, CUB_IMAGES, CUB_SEGMENTATIONS, BASE_MODELS
from data.dataloaders import vanillaCUBDataset, generateSegmentationMasks
from models.consts import NUM_CLASSES

from segmentation_models_pytorch import Unet, UnetPlusPlus, Linknet, FPN, Segformer

from tqdm import tqdm


In [ ]:





arch_encoder_pairings = { 
    # Unet : ["resnet34", "resnet18"],
    UnetPlusPlus : ["resnet34", "resnet18"],
    Linknet : ["resnet18"],
    FPN : ["resnet34"],
    Segformer : ["resnet18"],
}

def archToString(arch):
    if arch is Unet:
        return 'Unet'
    elif arch is UnetPlusPlus:
        return 'UnetPlusPlus'
    elif arch is Linknet:
        return 'Linknet'
    elif arch is FPN:
        return 'FPN'
    else:
        raise ValueError(f"Unknown architecture: {arch}")


def GENERATE_ALL_PREDICTIONS():
    for arch, encoders in arch_encoder_pairings.items(): 
        for encoder in encoders:
            model = arch(
                encoder_name=encoder,
                encoder_weights=None,
                in_channels=3,
                classes=NUM_CLASSES,
            )

            archstr = archToString(arch)
            model_weights_path = os.path.join(BASE_MODELS, f"{archstr}_{encoder}", "outputs", "best_model.pth")
            print(f"Model weights expected at: {model_weights_path} (exists={os.path.exists(model_weights_path)})")

            generateSegmentationMasks(
                DATASET = vanillaCUBDataset,
                model=model,
                model_weights_path=model_weights_path,
                image_dir=CUB_IMAGES,
                segmentation_dir=CUB_SEGMENTATIONS,
                model_name=f"{archstr}_{encoder}",
                batch_size=1,
                save_dir=BASE_MODELS_SEGMENTATIONS,
            )
    
    
def debug_print_paths():
    print("=== Path Debug ===")
    paths = {
        "CUB_IMAGES": CUB_IMAGES,
        "CUB_SEGMENTATIONS": CUB_SEGMENTATIONS,
        "BASE_MODELS_SEGMENTATIONS": BASE_MODELS_SEGMENTATIONS,
        "BASE_MODELS": BASE_MODELS,
    }
    for name, path in paths.items():
        print(f"{name}: {path} (exists={os.path.exists(path)})")
    print("==================")


def main(): 
    debug_print_paths()
    print("Going ahead with generating all masks...")
    GENERATE_ALL_PREDICTIONS()
    print("Generation complete.")
        

if __name__ == "__main__":
    main()

=== Path Debug ===
CUB_IMAGES: C:\Users\GAMER01\codeproj\fusionLearning\fusionLearning\data\CUBdata\CUB_200_2011\images (exists=True)
CUB_SEGMENTATIONS: C:\Users\GAMER01\codeproj\fusionLearning\fusionLearning\data\CUBdata\segmentations (exists=True)
BASE_MODELS_SEGMENTATIONS: C:\Users\GAMER01\codeproj\fusionLearning\images\segmentations (exists=True)
BASE_MODELS: C:\Users\GAMER01\codeproj\fusionLearning\fusionLearning\models (exists=True)
Going ahead with generating all masks...
Model weights expected at: C:\Users\GAMER01\codeproj\fusionLearning\fusionLearning\models\Unet_resnet34\outputs\best_model.pth (exists=True)
Using device: cuda
Dataset loaded with 11788 image-segmentation pairs
Segmentation masks for Unet_resnet34 already exist, skipping.
Model weights expected at: C:\Users\GAMER01\codeproj\fusionLearning\fusionLearning\models\Unet_resnet18\outputs\best_model.pth (exists=True)
Using device: cuda
Dataset loaded with 11788 image-segmentation pairs


Generated all segmentation masks for Unet_resnet18, saved to ./C:\Users\GAMER01\codeproj\fusionLearning\images\segmentations\Unet_resnet18
Model weights expected at: C:\Users\GAMER01\codeproj\fusionLearning\fusionLearning\models\UnetPlusPlus_resnet34\outputs\best_model.pth (exists=True)
Using device: cuda


RuntimeError: Error(s) in loading state_dict for UnetPlusPlus:
	size mismatch for segmentation_head.0.weight: copying a param with shape torch.Size([2, 16, 3, 3]) from checkpoint, the shape in current model is torch.Size([1, 16, 3, 3]).
	size mismatch for segmentation_head.0.bias: copying a param with shape torch.Size([2]) from checkpoint, the shape in current model is torch.Size([1]).